### Crystal Lattice (3D extension of the Chain-Spring model)

The system extends to a 3D cubic lattice of point masses, each of mass $m$, connected by ideal springs with spring constant $k$ and zero rest length. The boundaries are fixed. For a grid of size $N \times N \times N$, with equilibrium positions at $(x_{ijk}^0, y_{ijk}^0, z_{ijk}^0) = (i a, j a, k a)$ for $i,j,k = 0,\dots,N-1$, we define displacements $(u_{ijk}, v_{ijk}, w_{ijk})$ for interior points $i,j,k = 1,\dots,N-2$. The equations of motion decouple for $u$, $v$, and $w$:

\begin{cases}
    m \frac{d^2 u_{ijk}}{dt^2} + k\left(6u_{ijk} - u_{i-1,j,k} - u_{i+1,j,k} - u_{i,j-1,k} - u_{i,j+1,k} - u_{i,j,k-1} - u_{i,j,k+1}\right) = 0 \\
    m \frac{d^2 v_{ijk}}{dt^2} + k\left(6v_{ijk} - v_{i-1,j,k} - v_{i+1,j,k} - v_{i,j-1,k} - v_{i,j+1,k} - v_{i,j,k-1} - v_{i,j,k+1}\right) = 0 \\
    m \frac{d^2 w_{ijk}}{dt^2} + k\left(6w_{ijk} - w_{i-1,j,k} - w_{i+1,j,k} - w_{i,j-1,k} - w_{i,j+1,k} - w_{i,j,k-1} - w_{i,j,k+1}\right) = 0
\end{cases}

In matrix form, defining vectors $\mathbf{u}$, $\mathbf{v}$, and $\mathbf{w}$ by stacking all $u_{ijk}$, $v_{ijk}$, and $w_{ijk}$ in lexicographic order, the system becomes:

$$
    \frac{d^2}{dt^2}\begin{bmatrix} \mathbf{u} \\ \mathbf{v} \\ \mathbf{w} \end{bmatrix} + \frac{k}{m} \begin{bmatrix} \mathbf{K}_{3D} & \mathbf{0} & \mathbf{0} \\ \mathbf{0} & \mathbf{K}_{3D} & \mathbf{0} \\ \mathbf{0} & \mathbf{0} & \mathbf{K}_{3D} \end{bmatrix} \begin{bmatrix} \mathbf{u} \\ \mathbf{v} \\ \mathbf{w} \end{bmatrix} = \mathbf{0},
$$

where $\mathbf{K}_{3D}$ is the discrete 3D Laplacian matrix with Dirichlet boundary conditions.

In [1]:
import numpy as np
from scipy.sparse.linalg import LaplacianNd
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

In [2]:
def K_matrix_3D(N):

    return LaplacianNd(
        grid_shape = (N - 2, N - 2, N - 2),  # Only interior points
        boundary_conditions = 'dirichlet',
        dtype = np.float64
    )

def initial_3D(N, a):

    num_interior = (N - 2) ** 3
    
    # Random displacements for x, y, and z directions
    u_ini = (np.random.rand(num_interior) - 0.5) * 0.2 * a
    v_ini = (np.random.rand(num_interior) - 0.5) * 0.2 * a
    w_ini = (np.random.rand(num_interior) - 0.5) * 0.2 * a
    
    return (u_ini, v_ini, w_ini)

def system_rhs_3D(t, y, k, m, K):

    num_interior = len(y) // 6  # u, v, w, du/dt, dv/dt, dw/dt
    
    u = y[0 : num_interior]
    v = y[num_interior : 2 * num_interior]
    w = y[2*num_interior : 3 * num_interior]
    
    du_dt = y[3 * num_interior : 4 * num_interior]
    dv_dt = y[4 * num_interior : 5 * num_interior]
    dw_dt = y[5 * num_interior : 6 * num_interior]
    
    # Accelerations from linearized equations
    acc_u = (k / m) * (K @ u)
    acc_v = (k / m) * (K @ v)
    acc_w = (k / m) * (K @ w)
    
    dydt = np.concatenate([du_dt, dv_dt, dw_dt, acc_u, acc_v, acc_w])
    return dydt

def solver_3D(N, m, k, a, t_span):

    K = K_matrix_3D(N)
    u_ini, v_ini, w_ini = initial_3D(N, a)
    
    # Initial state: [u, v, w, du/dt, dv/dt, dw/dt]
    num_interior = (N - 2) ** 3
    y0 = np.concatenate([
        u_ini, v_ini, w_ini,
        np.zeros(num_interior),
        np.zeros(num_interior),
        np.zeros(num_interior)
    ])
    
    solution = solve_ivp(
        fun = lambda t, y: system_rhs_3D(t, y, k, m, K),
        t_span = t_span,
        y0 = y0,
        dense_output = True
    )
    
    return (solution, K)

def create_animation_3D(solution, N, a, output_file = 'Spring_Mass_System_3D.gif'):

    fps, dpi = 60, 120
    
    # Extract data
    t = solution.t
    y = solution.y
    num_interior = (N - 2) ** 3
    
    # Separate displacements
    u_history = y[0 : num_interior, :]
    v_history = y[num_interior : 2 * num_interior, :]
    w_history = y[2 * num_interior : 3 * num_interior, :]
    
    # Equilibrium positions
    X_eq, Y_eq, Z_eq = np.meshgrid(
        np.arange(1, N-1) * a,
        np.arange(1, N-1) * a,
        np.arange(1, N-1) * a,
        indexing = 'ij'
    )
    X_eq = X_eq.flatten()
    Y_eq = Y_eq.flatten()
    Z_eq = Z_eq.flatten()
    
    fig = plt.figure(figsize = (10, 8))
    ax = fig.add_subplot(111, projection = '3d')
    
    # Set limits
    ax.set_xlim(- 0.5, (N - 1) * a + 0.5)
    ax.set_ylim(- 0.5, (N - 1) * a + 0.5)
    ax.set_zlim(- 0.5, (N - 1) * a + 0.5)
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.set_title('3D Mass-Spring Lattice Oscillations')
    
    # Plot fixed boundary points (simplified - just the surface)
    boundary_points = []
    for i in [0, N - 1]:
        for j in range(N):
            for k in range(N):
                boundary_points.append([i * a, j * a, k * a])
    for j in [0, N - 1]:
        for i in range(1, N - 1):
            for k in range(N):
                boundary_points.append([i * a, j * a, k * a])
    for k in [0, N - 1]:
        for i in range(1, N - 1):
            for j in range(1, N - 1):
                boundary_points.append([i * a, j * a, k * a])
    
    boundary_points = np.array(boundary_points)
    ax.scatter(boundary_points[:, 0], boundary_points[:, 1], boundary_points[:, 2], c = 'k', s = 20, alpha = 0.3, marker = 's')
    
    # Plot movable points
    scatter = ax.scatter([], [], [], c = 'r', s = 30, alpha = 0.8)
    
    def init():
        scatter._offsets3d = (X_eq, Y_eq, Z_eq)
        return (scatter,)
    
    def update(frame):
        X_current = X_eq + u_history[:, frame]
        Y_current = Y_eq + v_history[:, frame]
        Z_current = Z_eq + w_history[:, frame]
        scatter._offsets3d = (X_current, Y_current, Z_current)
        return (scatter,)
    
    ani = FuncAnimation(
        fig = fig,
        func = update,
        frames = len(t),
        init_func = init,
        blit = True,
        interval = 1_000 / fps,
        cache_frame_data = False
    )
    
    print(f"Creating GIF animation...")
    writer = PillowWriter(fps = fps, bitrate = 2_000)
    ani.save(
        output_file,
        writer = writer,
        dpi = dpi,
        progress_callback = lambda i, n: print(f"\rFrame {i + 1}/{n} processed...", end = '')
    )
    
    plt.close(fig)

    print("Done!")
    return None

In [3]:
# Solve and animate the 3D system
N = 8 
m = 1.5
k = 4.5
a = 0.3
t_span = [0, 20]

solution_3D, K_3D = solver_3D(N, m, k, a, t_span)

print(f"System solved successfully!")
print(f"Number of interior points: {(N - 2) ** 3}")
print(f"Number of time steps: {len(solution_3D.t)}")

create_animation_3D(solution_3D, N, a)

System solved successfully!
Number of interior points: 216
Number of time steps: 111
Creating GIF animation...
Frame 111/111 processed...Done!


In [4]:
eigs_3D = (- K_3D.eigenvalues())[::-1]

print("Eigenvalues of K_3D (first 10):", eigs_3D[0 : 10])
print("\nNormal frequencies (first 10):", np.sqrt(eigs_3D[0 : 10] * k / m))

Eigenvalues of K_3D (first 10): [0.59418679 1.14914492 1.14914492 1.14914492 1.70410306 1.70410306
 1.70410306 1.95108266 1.95108266 1.95108266]

Normal frequencies (first 10): [1.3351256  1.8567269  1.8567269  1.8567269  2.26104161 2.26104161
 2.26104161 2.41934867 2.41934867 2.41934867]
